# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/Users/yunchaohe/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# Separate the feature matrix and class labels.
X = df.drop(columns=["Class"]).to_numpy(dtype=np.float32)
y = df["Class"].to_numpy(dtype=np.int32)

# Verify the shapes and label values.
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique labels:", np.unique(y))


X shape: (178, 13)
y shape: (178,)
Unique labels: [0 1 2]


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# Split the dataset into 70% training data and 30% test data.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Verify the resulting shapes and class distributions.
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("Training class distribution:", np.bincount(y_train))
print("Test class distribution:", np.bincount(y_test))

X_train shape: (124, 13)
X_test shape: (54, 13)
y_train shape: (124,)
y_test shape: (54,)
Training class distribution: [41 50 33]
Test class distribution: [18 21 15]


In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# Create and fit the scaler using only the training data.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to float32 for TensorFlow.
X_train_scaled = X_train_scaled.astype(np.float32)
X_test_scaled = X_test_scaled.astype(np.float32)

# Verify the results.
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

print("\nTraining feature means after scaling:")
print(np.mean(X_train_scaled, axis=0))

print("\nTraining feature standard deviations after scaling:")
print(np.std(X_train_scaled, axis=0))

X_train_scaled shape: (124, 13)
X_test_scaled shape: (54, 13)

Training feature means after scaling:
[ 2.9537946e-07 -1.8746622e-08  1.9323441e-07 -6.6814884e-08
  8.2970324e-08 -1.2473714e-07  1.5381843e-08  1.3459113e-07
  7.8831953e-08  2.3553449e-08  1.2317492e-08 -1.2305475e-07
  5.7681913e-08]

Training feature standard deviations after scaling:
[1.         1.         0.9999998  0.99999964 0.99999994 0.99999994
 1.0000001  1.0000001  1.0000001  1.         0.99999994 1.0000001
 0.99999994]


In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# Convert integer class labels to one-hot encoded labels.
y_train_encoded = to_categorical(y_train, num_classes=num_classes)
y_test_encoded = to_categorical(y_test, num_classes=num_classes)

# Convert explicitly to float32 for TensorFlow.
y_train_encoded = y_train_encoded.astype(np.float32)
y_test_encoded = y_test_encoded.astype(np.float32)

# Verify the results.
print("y_train_encoded shape:", y_train_encoded.shape)
print("y_test_encoded shape:", y_test_encoded.shape)

print("\nFirst five original training labels:")
print(y_train[:5])

print("\nFirst five one-hot encoded training labels:")
print(y_train_encoded[:5])


y_train_encoded shape: (124, 3)
y_test_encoded shape: (54, 3)

First five original training labels:
[0 0 0 0 1]

First five one-hot encoded training labels:
[[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]]


In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# Define the baseline DNN model.
baseline_model = Sequential([
    Dense(64, activation="relu", input_shape=(num_features,)),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax")
])

# Display the model architecture and parameter counts.
baseline_model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# Compile the baseline model.
baseline_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the model.
history = baseline_model.fit(
    X_train_scaled,
    y_train_encoded,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
13/13 [==============================] - 0s 4ms/step - loss: 0.9836 - accuracy: 0.5960 - val_loss: 0.8454 - val_accuracy: 0.7600
Epoch 2/20
13/13 [==============================] - 0s 917us/step - loss: 0.7200 - accuracy: 0.9495 - val_loss: 0.6481 - val_accuracy: 0.9200
Epoch 3/20
13/13 [==============================] - 0s 962us/step - loss: 0.5332 - accuracy: 0.9798 - val_loss: 0.4788 - val_accuracy: 0.9600
Epoch 4/20
13/13 [==============================] - 0s 957us/step - loss: 0.3822 - accuracy: 0.9899 - val_loss: 0.3585 - val_accuracy: 0.9600
Epoch 5/20
13/13 [==============================] - 0s 916us/step - loss: 0.2716 - accuracy: 0.9899 - val_loss: 0.2710 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 892us/step - loss: 0.1929 - accuracy: 0.9899 - val_loss: 0.2092 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 886us/step - loss: 0.1430 - accuracy: 0.9899 - val_loss: 0.1677 - val_accuracy: 0.9600
Epoch 8/

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# Evaluate the baseline model on the test set.
test_loss, test_accuracy = baseline_model.evaluate(
    X_test_scaled,
    y_test_encoded,
    verbose=0
)

# Predict class probabilities and convert them to class labels.
y_pred_prob = baseline_model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

# Print evaluation results.
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Test loss: 0.0807
Test accuracy: 0.9630 (96.30%)

Classification report:
              precision    recall  f1-score   support

           0     0.9474    1.0000    0.9730        18
           1     1.0000    0.9048    0.9500        21
           2     0.9375    1.0000    0.9677        15

    accuracy                         0.9630        54
   macro avg     0.9616    0.9683    0.9636        54
weighted avg     0.9651    0.9630    0.9626        54

Confusion matrix:
[[18  0  0]
 [ 1 19  1]
 [ 0  0 15]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# Convert the trained Keras model to a float32 TFLite model.
converter = tf.lite.TFLiteConverter.from_keras_model(baseline_model)
tflite_base_model = converter.convert()

# Save the TFLite model to a file.
base_model_path = "model_base.tflite"

with open(base_model_path, "wb") as f:
    f.write(tflite_base_model)

# Calculate and print the file size.
import os

base_model_size_bytes = os.path.getsize(base_model_path)
base_model_size_kb = base_model_size_bytes / 1024

print(f"Float32 TFLite model saved as: {base_model_path}")
print(f"Model size: {base_model_size_bytes} bytes")
print(f"Model size: {base_model_size_kb:.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpxe3bih93/assets


INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpxe3bih93/assets


Float32 TFLite model saved as: model_base.tflite
Model size: 14408 bytes
Model size: 14.07 KB


2026-07-27 11:02:11.451790: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-27 11:02:11.451810: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-27 11:02:11.451987: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpxe3bih93
2026-07-27 11:02:11.452280: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-27 11:02:11.452283: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpxe3bih93
2026-07-27 11:02:11.453070: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-07-27 11:02:11.453406: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-27 11:02:11.466212: I tensorflow/cc/saved_model/loader.

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [17]:
import os

def file_size_kb(filename):
    """Return the size of a file in kilobytes."""
    return os.path.getsize(filename) / 1024

def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        # Enable default TFLite optimizations, including quantization.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        # Use representative training samples to calibrate activation ranges.
        converter.representative_dataset = lambda: representative_data_gen(
            X_train_scaled,
            num_samples=100
        )

        # Require all model operations to use built-in int8 kernels.
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS_INT8
        ]

        # Make the external model input and output tensors int8.
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        # Enable default TFLite optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

        # Store eligible model weights using float16.
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        # Enable dynamic range quantization.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    tflite_quantized_model = converter.convert()

    with open(filename, "wb") as f:
        f.write(tflite_quantized_model)

    print(f"\nQuantized model saved as: {filename}")

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    # Load the saved TFLite model and allocate its tensors.
    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    # Get information about the model's input and output tensors.
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_detail = input_details[0]
    output_detail = output_details[0]

    print("Input tensor details:")
    print(
        " shape =", input_detail["shape"],
        "dtype =", input_detail["dtype"],
        "quantization =", input_detail["quantization"]
    )

    print("Output tensor details:")
    print(
        " shape =", output_detail["shape"],
        "dtype =", output_detail["dtype"],
        "quantization =", output_detail["quantization"]
    )

    predictions = []

    # Run inference one test sample at a time.
    for sample in X_test:
        # Add the batch dimension: (13,) -> (1, 13).
        input_sample = np.expand_dims(sample, axis=0).astype(np.float32)

        # Quantize the input if the TFLite model expects integer input.
        if np.issubdtype(input_detail["dtype"], np.integer):
            input_scale, input_zero_point = input_detail["quantization"]

            if input_scale == 0:
                raise ValueError("Input quantization scale cannot be zero.")

            input_sample = np.round(
                input_sample / input_scale + input_zero_point
            )

            # Clip values to the valid range of the input integer type.
            input_type_range = np.iinfo(input_detail["dtype"])
            input_sample = np.clip(
                input_sample,
                input_type_range.min,
                input_type_range.max
            ).astype(input_detail["dtype"])

        else:
            # Float models normally expect float32 input.
            input_sample = input_sample.astype(input_detail["dtype"])

        # Copy the input into the TFLite input tensor and run inference.
        interpreter.set_tensor(input_detail["index"], input_sample)
        interpreter.invoke()

        # Retrieve the model output.
        output_sample = interpreter.get_tensor(output_detail["index"])

        # Dequantize the output if it is stored as an integer tensor.
        if np.issubdtype(output_detail["dtype"], np.integer):
            output_scale, output_zero_point = output_detail["quantization"]

            if output_scale == 0:
                raise ValueError("Output quantization scale cannot be zero.")

            output_sample = (
                output_sample.astype(np.float32) - output_zero_point
            ) * output_scale

        # Select the class with the highest output probability/score.
        predicted_class = np.argmax(output_sample, axis=1)[0]
        predictions.append(predicted_class)

    # Convert predictions and one-hot labels to integer class arrays.
    y_pred = np.array(predictions, dtype=np.int32)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    # print classification_report and confusion_matrix <--#
    quantized_accuracy = np.mean(y_pred == y_true)

    print(
        f"{quant_type.upper()} test accuracy: "
        f"{quantized_accuracy:.4f} "
        f"({quantized_accuracy * 100:.2f}%)"
    )

    print(f"\n{quant_type.upper()} classification report:")
    print(classification_report(y_true, y_pred, digits=4))

    print(f"{quant_type.upper()} confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

    return {
        "quant_type": quant_type,
        "filename": filename,
        "model_size_kb": file_size_kb(filename),
        "accuracy": quantized_accuracy,
        "y_true": y_true,
        "y_pred": y_pred
    }


In [18]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

# Create and evaluate the Full Integer Quantized model.
int8_results = quantize_and_evaluate(
    model=baseline_model,
    X_test=X_test_scaled,
    y_test_cat=y_test_encoded,
    quant_type="int8",
    filename="model_int8.tflite"
)

# Create and evaluate the Float16 Quantized model.
float16_results = quantize_and_evaluate(
    model=baseline_model,
    X_test=X_test_scaled,
    y_test_cat=y_test_encoded,
    quant_type="float16",
    filename="model_float16.tflite"
)

# Create and evaluate the Dynamic Range Quantized model.
dynamic_results = quantize_and_evaluate(
    model=baseline_model,
    X_test=X_test_scaled,
    y_test_cat=y_test_encoded,
    quant_type="dynamic",
    filename="model_dynamic.tflite"
)


INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpsf06f3ey/assets


INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpsf06f3ey/assets



Quantized model saved as: model_int8.tflite
Input tensor details:
 shape = [ 1 13] dtype = <class 'numpy.int8'> quantization = (0.030695566907525063, -6)
Output tensor details:
 shape = [1 3] dtype = <class 'numpy.int8'> quantization = (0.00390625, -128)

INT8 TFLite model size: 5.74 KB
INT8 test accuracy: 0.9630 (96.30%)

INT8 classification report:
              precision    recall  f1-score   support

           0     0.9474    1.0000    0.9730        18
           1     1.0000    0.9048    0.9500        21
           2     0.9375    1.0000    0.9677        15

    accuracy                         0.9630        54
   macro avg     0.9616    0.9683    0.9636        54
weighted avg     0.9651    0.9630    0.9626        54

INT8 confusion matrix:
[[18  0  0]
 [ 1 19  1]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpvcqo4f9v/assets


/Users/yunchaohe/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-07-27 11:30:48.139229: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-27 11:30:48.139246: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-27 11:30:48.139359: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpsf06f3ey
2026-07-27 11:30:48.139651: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-27 11:30:48.139654: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpsf06f3ey
2026-07-27 11:30:48.140565: I tensorflow/cc/saved_model/loader.cc:233


Quantized model saved as: model_float16.tflite
Input tensor details:
 shape = [ 1 13] dtype = <class 'numpy.float32'> quantization = (0.0, 0)
Output tensor details:
 shape = [1 3] dtype = <class 'numpy.float32'> quantization = (0.0, 0)

FLOAT16 TFLite model size: 8.95 KB
FLOAT16 test accuracy: 0.9630 (96.30%)

FLOAT16 classification report:
              precision    recall  f1-score   support

           0     0.9474    1.0000    0.9730        18
           1     1.0000    0.9048    0.9500        21
           2     0.9375    1.0000    0.9677        15

    accuracy                         0.9630        54
   macro avg     0.9616    0.9683    0.9636        54
weighted avg     0.9651    0.9630    0.9626        54

FLOAT16 confusion matrix:
[[18  0  0]
 [ 1 19  1]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpxd3nm70a/assets


INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpxd3nm70a/assets



Quantized model saved as: model_dynamic.tflite
Input tensor details:
 shape = [ 1 13] dtype = <class 'numpy.float32'> quantization = (0.0, 0)
Output tensor details:
 shape = [1 3] dtype = <class 'numpy.float32'> quantization = (0.0, 0)

DYNAMIC TFLite model size: 8.17 KB
DYNAMIC test accuracy: 0.9630 (96.30%)

DYNAMIC classification report:
              precision    recall  f1-score   support

           0     0.9474    1.0000    0.9730        18
           1     1.0000    0.9048    0.9500        21
           2     0.9375    1.0000    0.9677        15

    accuracy                         0.9630        54
   macro avg     0.9616    0.9683    0.9636        54
weighted avg     0.9651    0.9630    0.9626        54

DYNAMIC confusion matrix:
[[18  0  0]
 [ 1 19  1]
 [ 0  0 15]]


2026-07-27 11:30:48.495256: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-27 11:30:48.495267: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-27 11:30:48.495346: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpxd3nm70a
2026-07-27 11:30:48.495602: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-27 11:30:48.495605: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpxd3nm70a
2026-07-27 11:30:48.496348: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-27 11:30:48.507363: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpxd3nm70a
2026-07-

## Problem 1 - Part (c)

### Pruning

In [21]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

# Define pruning training parameters.
pruning_epochs = 10
pruning_batch_size = 8

# Approximate total number of pruning training steps.
steps_per_epoch = int(np.ceil(len(X_train_scaled) / pruning_batch_size))
end_step = steps_per_epoch * pruning_epochs

# Define the polynomial sparsity schedule.
pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

print("Training samples:", len(X_train_scaled))
print("Steps per epoch:", steps_per_epoch)
print("Pruning end step:", end_step)

Training samples: 124
Steps per epoch: 16
Pruning end step: 160


In [22]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

# Build a DNN in which every Dense layer is wrapped for pruning.
pruned_model = Sequential([
    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(
            64,
            activation="relu",
            input_shape=(num_features,)
        ),
        pruning_schedule=pruning_schedule
    ),

    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(32, activation="relu"),
        pruning_schedule=pruning_schedule
    ),

    tfmot.sparsity.keras.prune_low_magnitude(
        Dense(num_classes, activation="softmax"),
        pruning_schedule=pruning_schedule
    )
])

pruned_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 6 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 7 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 8 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [23]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

# Compile the pruning-aware model.
pruned_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# This callback updates the pruning step after each training batch.
pruning_callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

# Train the pruning-aware model.
pruned_history = pruned_model.fit(
    X_train_scaled,
    y_train_encoded,
    epochs=pruning_epochs,
    batch_size=pruning_batch_size,
    validation_split=0.2,
    callbacks=pruning_callbacks,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 1s 4ms/step - loss: 1.1126 - accuracy: 0.3535 - val_loss: 0.9153 - val_accuracy: 0.5200
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7918 - accuracy: 0.7071 - val_loss: 0.6805 - val_accuracy: 0.8400
Epoch 3/10
13/13 [==============================] - 0s 975us/step - loss: 0.5827 - accuracy: 0.9293 - val_loss: 0.4916 - val_accuracy: 0.9600
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.4157 - accuracy: 0.9596 - val_loss: 0.3526 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.2909 - accuracy: 0.9697 - val_loss: 0.2466 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.2065 - accuracy: 0.9798 - val_loss: 0.1838 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.1511 - accuracy: 0.9798 - val_loss: 0.1414 - val_accuracy: 1.0000
Epoch 8/10
13/13 [

In [24]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

# Remove pruning wrappers and pruning-specific training variables.
stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

# Display the stripped model structure.
stripped_pruned_model.summary()

# Convert the stripped model to a standard float32 TFLite model.
converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
tflite_pruned_model = converter.convert()

# Save the converted TFLite model.
pruned_model_path = "model_pruned.tflite"

with open(pruned_model_path, "wb") as f:
    f.write(tflite_pruned_model)

# Report the final TFLite file size.
pruned_model_size_kb = file_size_kb(pruned_model_path)

print(f"Pruned TFLite model saved as: {pruned_model_path}")
print(f"Pruned TFLite model size: {pruned_model_size_kb:.2f} KB")


Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 64)                896       
                                                                 
 dense_7 (Dense)             (None, 32)                2080      
                                                                 
 dense_8 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpz3ybl0mf/assets


INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpz3ybl0mf/assets


Pruned TFLite model saved as: model_pruned.tflite
Pruned TFLite model size: 14.14 KB


2026-07-27 17:01:08.669600: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-27 17:01:08.669615: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-27 17:01:08.669707: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpz3ybl0mf
2026-07-27 17:01:08.669937: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-27 17:01:08.669940: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpz3ybl0mf
2026-07-27 17:01:08.671461: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-27 17:01:08.676764: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpz3ybl0mf
2026-07-

In [25]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

# Generate class probabilities using the stripped pruning model.
y_pruned_prob = stripped_pruned_model.predict(
    X_test_scaled,
    verbose=0
)

# Convert predicted probabilities and one-hot labels to integer classes.
y_pruned_pred = np.argmax(y_pruned_prob, axis=1)
y_pruned_true = np.argmax(y_test_encoded, axis=1)

# Calculate accuracy.
pruned_accuracy = np.mean(y_pruned_pred == y_pruned_true)

print(
    f"Pruned model test accuracy: "
    f"{pruned_accuracy:.4f} ({pruned_accuracy * 100:.2f}%)"
)

print("\nPruned model classification report:")
print(
    classification_report(
        y_pruned_true,
        y_pruned_pred,
        digits=4
    )
)

print("Pruned model confusion matrix:")
print(confusion_matrix(y_pruned_true, y_pruned_pred))

Pruned model test accuracy: 0.9630 (96.30%)

Pruned model classification report:
              precision    recall  f1-score   support

           0     0.9474    1.0000    0.9730        18
           1     0.9524    0.9524    0.9524        21
           2     1.0000    0.9333    0.9655        15

    accuracy                         0.9630        54
   macro avg     0.9666    0.9619    0.9636        54
weighted avg     0.9639    0.9630    0.9629        54

Pruned model confusion matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]


In [26]:
total_kernel_weights = 0
total_zero_weights = 0

for layer in stripped_pruned_model.layers:
    weights = layer.get_weights()

    if not weights:
        continue

    kernel = weights[0]
    zero_count = np.sum(kernel == 0)
    weight_count = kernel.size
    layer_sparsity = zero_count / weight_count

    total_zero_weights += zero_count
    total_kernel_weights += weight_count

    print(
        f"{layer.name}: "
        f"{zero_count}/{weight_count} zero weights, "
        f"sparsity = {layer_sparsity * 100:.2f}%"
    )

overall_sparsity = total_zero_weights / total_kernel_weights

print(
    f"\nOverall kernel sparsity: "
    f"{overall_sparsity * 100:.2f}%"
)

dense_6: 574/832 zero weights, sparsity = 68.99%
dense_7: 1412/2048 zero weights, sparsity = 68.95%
dense_8: 66/96 zero weights, sparsity = 68.75%

Overall kernel sparsity: 68.95%


## Problem 1 - Part (d)

### Knowledge Distillation

In [27]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

# Define the smaller student model.
student_model = Sequential([
    Dense(
        32,
        activation="relu",
        input_shape=(num_features,)
    ),
    Dense(16, activation="relu"),
    Dense(num_classes, activation="softmax")
])

# Display the student architecture and parameter count.
student_model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 32)                448       
                                                                 
 dense_10 (Dense)            (None, 16)                528       
                                                                 
 dense_11 (Dense)            (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [28]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

# Use the trained baseline model as the teacher.
teacher_model = baseline_model

# Generate teacher soft labels for the training samples.
teacher_soft_labels = teacher_model.predict(
    X_train_scaled,
    verbose=0
).astype(np.float32)

# Verify the soft-label output.
print("Teacher soft labels shape:", teacher_soft_labels.shape)
print("\nFirst five teacher soft labels:")
print(teacher_soft_labels[:5])

print("\nProbability sums for the first five samples:")
print(np.sum(teacher_soft_labels[:5], axis=1))

Teacher soft labels shape: (124, 3)

First five teacher soft labels:
[[9.9699080e-01 1.9353904e-03 1.0738499e-03]
 [9.9636769e-01 2.6116038e-03 1.0206511e-03]
 [9.8700315e-01 7.4277869e-03 5.5690897e-03]
 [9.9844861e-01 9.3874801e-04 6.1259756e-04]
 [8.5403966e-03 9.8970664e-01 1.7529747e-03]]

Probability sums for the first five samples:
[1.         0.99999994 1.         1.         1.        ]


In [29]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# Concatenate hard labels and teacher soft labels.
y_train_combined = np.concatenate(
    [y_train_encoded, teacher_soft_labels],
    axis=1
).astype(np.float32)

alpha = 0.5

def distillation_loss(y_true_combined, y_pred):
    # Split the first three columns as hard labels.
    y_true_hard = y_true_combined[:, :num_classes]

    # Split the last three columns as teacher soft labels.
    y_true_soft = y_true_combined[:, num_classes:]

    # Calculate loss against the ground-truth hard labels.
    hard_label_loss = tf.keras.losses.categorical_crossentropy(
        y_true_hard,
        y_pred
    )

    # Calculate loss against the teacher's soft labels.
    soft_label_loss = tf.keras.losses.categorical_crossentropy(
        y_true_soft,
        y_pred
    )

    # Combine the hard-label and soft-label losses.
    return alpha * hard_label_loss + (1.0 - alpha) * soft_label_loss


# Verify the combined-label shape.
print("Hard labels shape:", y_train_encoded.shape)
print("Teacher soft labels shape:", teacher_soft_labels.shape)
print("Combined labels shape:", y_train_combined.shape)

print("\nFirst combined label:")
print(y_train_combined[0])
    

Hard labels shape: (124, 3)
Teacher soft labels shape: (124, 3)
Combined labels shape: (124, 6)

First combined label:
[1.         0.         0.         0.9969908  0.00193539 0.00107385]


In [30]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

# Compile the student model using the custom distillation loss.
student_model.compile(
    optimizer="adam",
    loss=distillation_loss
)

# Train the student using the combined hard and soft labels.
student_history = student_model.fit(
    X_train_scaled,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 0s 4ms/step - loss: 1.1274 - val_loss: 0.9986
Epoch 2/10
13/13 [==============================] - 0s 916us/step - loss: 0.9097 - val_loss: 0.8497
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7468 - val_loss: 0.7217
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.6145 - val_loss: 0.6070
Epoch 5/10
13/13 [==============================] - 0s 956us/step - loss: 0.5024 - val_loss: 0.5075
Epoch 6/10
13/13 [==============================] - 0s 905us/step - loss: 0.4118 - val_loss: 0.4203
Epoch 7/10
13/13 [==============================] - 0s 866us/step - loss: 0.3357 - val_loss: 0.3549
Epoch 8/10
13/13 [==============================] - 0s 884us/step - loss: 0.2777 - val_loss: 0.3017
Epoch 9/10
13/13 [==============================] - 0s 870us/step - loss: 0.2313 - val_loss: 0.2628
Epoch 10/10
13/13 [==============================] - 0s 894us/step - loss: 0.1951 - val_loss: 0.2313


In [31]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

# Convert the trained student model to a float32 TFLite model.
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd_model = converter.convert()

# Save the TFLite student model.
kd_model_path = "model_kd.tflite"

with open(kd_model_path, "wb") as f:
    f.write(tflite_kd_model)

# Print the saved model size.
kd_model_size_kb = file_size_kb(kd_model_path)

print(f"Knowledge-distilled student model saved as: {kd_model_path}")
print(f"Knowledge-distilled TFLite model size: {kd_model_size_kb:.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmp3ldn0u5g/assets


INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmp3ldn0u5g/assets


Knowledge-distilled student model saved as: model_kd.tflite
Knowledge-distilled TFLite model size: 6.12 KB


2026-07-27 17:31:58.053649: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-27 17:31:58.053665: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-27 17:31:58.053746: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmp3ldn0u5g
2026-07-27 17:31:58.054119: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-27 17:31:58.054122: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmp3ldn0u5g
2026-07-27 17:31:58.054896: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-27 17:31:58.065659: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmp3ldn0u5g
2026-07-

In [32]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

# Obtain the student's class probabilities on the test set.
y_student_prob = student_model.predict(
    X_test_scaled,
    verbose=0
)

# Convert probabilities and one-hot labels to integer class labels.
y_student_pred = np.argmax(y_student_prob, axis=1)
y_student_true = np.argmax(y_test_encoded, axis=1)

# Calculate test accuracy.
student_accuracy = np.mean(y_student_pred == y_student_true)

print(
    f"Knowledge-distilled student test accuracy: "
    f"{student_accuracy:.4f} ({student_accuracy * 100:.2f}%)"
)

print("\nKnowledge-distilled student classification report:")
print(
    classification_report(
        y_student_true,
        y_student_pred,
        digits=4
    )
)

print("Knowledge-distilled student confusion matrix:")
print(confusion_matrix(y_student_true, y_student_pred))

Knowledge-distilled student test accuracy: 0.9630 (96.30%)

Knowledge-distilled student classification report:
              precision    recall  f1-score   support

           0     0.9474    1.0000    0.9730        18
           1     0.9524    0.9524    0.9524        21
           2     1.0000    0.9333    0.9655        15

    accuracy                         0.9630        54
   macro avg     0.9666    0.9619    0.9636        54
weighted avg     0.9639    0.9630    0.9629        54

Knowledge-distilled student confusion matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [33]:
# 3: Implement the proposed KD + Full INT8 solution.

# Create a TFLite converter from the trained knowledge-distilled student.
kd_int8_converter = tf.lite.TFLiteConverter.from_keras_model(student_model)

# Enable post-training optimization.
kd_int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Use representative training samples to calibrate activation ranges.
kd_int8_converter.representative_dataset = (
    lambda: representative_data_gen(
        X_train_scaled,
        num_samples=100
    )
)

# Require the model to use only built-in INT8 operators.
kd_int8_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

# Use INT8 for the model's external input and output tensors.
kd_int8_converter.inference_input_type = tf.int8
kd_int8_converter.inference_output_type = tf.int8

# Convert the Float32 KD student to a Full INT8 TFLite model.
tflite_kd_int8_model = kd_int8_converter.convert()

# Save the compressed model.
kd_int8_model_path = "model_kd_int8.tflite"

with open(kd_int8_model_path, "wb") as f:
    f.write(tflite_kd_int8_model)

print(f"KD + INT8 model saved as: {kd_int8_model_path}")

INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpzpl625wb/assets


INFO:tensorflow:Assets written to: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpzpl625wb/assets


KD + INT8 model saved as: model_kd_int8.tflite


/Users/yunchaohe/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-07-27 23:26:28.825807: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-27 23:26:28.825825: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-27 23:26:28.825977: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpzpl625wb
2026-07-27 23:26:28.826242: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-27 23:26:28.826248: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/vd/_brkz_g91_g7w7y_bryvtkhh0000gn/T/tmpzpl625wb
2026-07-27 23:26:28.827124: I tensorflow/cc/saved_model/loader.cc:233

In [34]:
# 4: Evaluate the KD + INT8 TFLite model.

# Report the TFLite file size.
kd_int8_model_size_kb = file_size_kb(kd_int8_model_path)

print(f"KD + INT8 TFLite model size: {kd_int8_model_size_kb:.2f} KB")

# Load the saved TFLite model.
kd_int8_interpreter = tf.lite.Interpreter(
    model_path=kd_int8_model_path
)

# Allocate memory for input, intermediate, and output tensors.
kd_int8_interpreter.allocate_tensors()

# Get model input and output tensor information.
input_detail = kd_int8_interpreter.get_input_details()[0]
output_detail = kd_int8_interpreter.get_output_details()[0]

print("\nInput tensor:")
print(
    "shape =", input_detail["shape"],
    "dtype =", input_detail["dtype"],
    "quantization =", input_detail["quantization"]
)

print("Output tensor:")
print(
    "shape =", output_detail["shape"],
    "dtype =", output_detail["dtype"],
    "quantization =", output_detail["quantization"]
)

# Read the quantization parameters.
input_scale, input_zero_point = input_detail["quantization"]
output_scale, output_zero_point = output_detail["quantization"]

if input_scale == 0:
    raise ValueError("Input quantization scale cannot be zero.")

if output_scale == 0:
    raise ValueError("Output quantization scale cannot be zero.")

# Determine the valid range of the input integer type.
input_type_range = np.iinfo(input_detail["dtype"])

kd_int8_predictions = []

# Run TFLite inference one test sample at a time.
for sample in X_test_scaled:
    # Add a batch dimension: (13,) -> (1, 13).
    input_sample = np.expand_dims(
        sample,
        axis=0
    ).astype(np.float32)

    # Quantize the Float32 input to INT8.
    quantized_input = np.round(
        input_sample / input_scale + input_zero_point
    )

    # Clip to the valid INT8 range and convert to the required dtype.
    quantized_input = np.clip(
        quantized_input,
        input_type_range.min,
        input_type_range.max
    ).astype(input_detail["dtype"])

    # Set the input tensor and run inference.
    kd_int8_interpreter.set_tensor(
        input_detail["index"],
        quantized_input
    )
    kd_int8_interpreter.invoke()

    # Retrieve the quantized INT8 output.
    quantized_output = kd_int8_interpreter.get_tensor(
        output_detail["index"]
    )

    # Dequantize the output back to Float32.
    dequantized_output = (
        quantized_output.astype(np.float32) - output_zero_point
    ) * output_scale

    # Select the class with the highest output score.
    predicted_class = np.argmax(
        dequantized_output,
        axis=1
    )[0]

    kd_int8_predictions.append(predicted_class)

# Convert predictions and one-hot test labels to integer arrays.
y_kd_int8_pred = np.array(
    kd_int8_predictions,
    dtype=np.int32
)

y_kd_int8_true = np.argmax(
    y_test_encoded,
    axis=1
)

# Calculate classification accuracy.
kd_int8_accuracy = np.mean(
    y_kd_int8_pred == y_kd_int8_true
)

print(
    f"\nKD + INT8 test accuracy: "
    f"{kd_int8_accuracy:.4f} "
    f"({kd_int8_accuracy * 100:.2f}%)"
)

print("\nKD + INT8 classification report:")
print(
    classification_report(
        y_kd_int8_true,
        y_kd_int8_pred,
        digits=4
    )
)

print("KD + INT8 confusion matrix:")
print(
    confusion_matrix(
        y_kd_int8_true,
        y_kd_int8_pred
    )
)

KD + INT8 TFLite model size: 3.66 KB

Input tensor:
shape = [ 1 13] dtype = <class 'numpy.int8'> quantization = (0.030695566907525063, -6)
Output tensor:
shape = [1 3] dtype = <class 'numpy.int8'> quantization = (0.00390625, -128)

KD + INT8 test accuracy: 0.9630 (96.30%)

KD + INT8 classification report:
              precision    recall  f1-score   support

           0     0.9474    1.0000    0.9730        18
           1     0.9524    0.9524    0.9524        21
           2     1.0000    0.9333    0.9655        15

    accuracy                         0.9630        54
   macro avg     0.9666    0.9619    0.9636        54
weighted avg     0.9639    0.9630    0.9629        54

KD + INT8 confusion matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  1 14]]


# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
